In [1]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from pathlib import Path

/home/eren/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class ToxicModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = AutoModel.from_pretrained("dbmdz/bert-base-turkish-cased")
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        x = self.dropout(cls)
        x = self.fc(x)
        return x

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained("dbmdz/bert-base-turkish-cased")

In [ ]:
def find_checkpoint_path():
    candidates = [
        Path("best_toxic_model.pt"),
        Path("../best_toxic_model.pt"),
    ]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("best_toxic_model.pt bulunamadi. Dosyayi test.py ile ayni klasore ya da bir ust klasore koyun.")


def load_model_and_threshold(device):
    checkpoint_path = find_checkpoint_path()
    checkpoint = torch.load(checkpoint_path, map_location=device)

    model = ToxicModel().to(device)
    if isinstance(checkpoint, dict) and "model_state_dict" in checkpoint:
        model.load_state_dict(checkpoint["model_state_dict"])
        threshold = float(checkpoint.get("threshold", 0.5))
    else:
        model.load_state_dict(checkpoint)
        threshold = 0.5

    return model, threshold

# 4 -> A
# 0 -> O
# 1 -> I
# 5 -> S
# 2 -> iki
# 3 -> E
# 6 -> G
def leetspeak_to_normal(text):
    leet_dict = {
        'a': ['4', '@', "/\\"],
        'o': ['0'],
        'i': ['1','|'],
        'l': ['1'],
        's': ['5'],
        'iki': ['2'],
        'e': ['3'],
        'g': ['6']
    }
    for normal_char, leet_variants in leet_dict.items():
        for leet in leet_variants:
            text = text.replace(leet, normal_char)
    return text

Modelin ve thresholdun yüklenmesi

In [4]:
model, best_threshold = load_model_and_threshold(device)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5116.92it/s]
BertModel LOAD REPORT from: dbmdz/bert-base-turkish-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
def predict_text(model, tokenizer, text, device, max_length=128, threshold=0.5):
    model.eval()
    tokens = tokenizer(
        text,
        padding="max_length",
        truncation=True,
        max_length=max_length,
        return_tensors="pt"
    )
    input_ids = tokens["input_ids"].to(device)
    attention_mask = tokens["attention_mask"].to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask)
        prob = torch.sigmoid(logits).item()
        print(f"Text: {text}\nToxicity Probability: {prob:.4f}")
        return 1 if prob > threshold else 0

In [23]:
cumle = "Çok güzel bir filmdi izledim, 41k kaliteli ve harika bir senaryoya sahipti!"
toxic_cumle = "Bu film tam bir b0k, aptal salak hiç beğenmedim ve zaman kaybıydı."
print(leetspeak_to_normal(toxic_cumle.lower()))
print(predict_text(model, tokenizer, leetspeak_to_normal(toxic_cumle.lower()), device, threshold=best_threshold))

print(leetspeak_to_normal(cumle.lower()))
print(predict_text(model, tokenizer, leetspeak_to_normal(cumle.lower()), device, threshold=best_threshold))

bu film tam bir bok, aptal salak hiç beğenmedim ve zaman kaybıydı.
Text: bu film tam bir bok, aptal salak hiç beğenmedim ve zaman kaybıydı.
Toxicity Probability: 1.0000
1
çok güzel bir filmdi izledim, aik kaliteli ve harika bir senaryoya sahipti!
Text: çok güzel bir filmdi izledim, aik kaliteli ve harika bir senaryoya sahipti!
Toxicity Probability: 0.0000
0
